In [1]:
!pip install google-api-python-client pandas

In [2]:
from googleapiclient.discovery import build
import pandas as pd

# Scrape the Data using Youtube API

In [3]:
API_KEY = "#######################################"

youtube = build('youtube', 'v3', developerKey=API_KEY)

video_ids = ["-U5EWezaQos",]

all_comments = []
MAX_COMMENTS = 10000

for video_id in video_ids:
    print(f"Fetching comments for video: {video_id}")
    
    next_page_token = None

    while True:
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100,
            pageToken=next_page_token,
            textFormat="plainText"
        )

        response = request.execute()

        for item in response["items"]:
            if len(all_comments) >= MAX_COMMENTS:
                break

            comment = item["snippet"]["topLevelComment"]["snippet"]

            all_comments.append({
                "video_id": video_id,
                "author": comment["authorDisplayName"],
                "comment_text": comment["textDisplay"],
                "like_count": comment["likeCount"],
                "published_at": comment["publishedAt"]
            })

        if len(all_comments) >= MAX_COMMENTS:
            break

        next_page_token = response.get("nextPageToken")

        if not next_page_token:
            break

print(f"Total comments collected: {len(all_comments)}")

Fetching comments for video: -U5EWezaQos
Total comments collected: 10000


In [12]:
df = pd.DataFrame(all_comments)

In [13]:
df

,video_id,author,comment_text,like_count,published_at
0,-U5EWezaQos,@shaiknasirofficial,1.9M\nDislikes 😂,0,2026-02-14T03:50:25Z
1,-U5EWezaQos,@Hirokojima-t1r,On 14 February 2025 the dislikes are 1.9 million,0,2026-02-13T19:37:07Z
2,-U5EWezaQos,@VincenzoS7S,1.9M dislikes,0,2026-02-11T22:45:34Z
3,-U5EWezaQos,@jetrho3447,kitne dislikes hai ??😂,0,2026-02-06T08:09:27Z
4,-U5EWezaQos,@Jospehsiju,You sicken me,0,2026-02-05T23:10:35Z
...,...,...,...,...,...
9995,-U5EWezaQos,@rockysingh837,Pleasw inko fully ignore karo indians,0,2020-08-15T02:38:02Z
9996,-U5EWezaQos,@AmanSingh-cb4qy,Job done,0,2020-08-15T02:37:57Z
9997,-U5EWezaQos,@ATR-u6e,Uparbala teri sath nehi hai pranati kahike,0,2020-08-15T02:37:46Z
9998,-U5EWezaQos,@kanusharmaphotography,Most bakwas.,0,2020-08-15T02:37:20Z


# Data Cleaning

## Drop the video_id because the data was taken from a single video.

In [14]:
df = df.drop(columns=["video_id"])

In [15]:
df

,author,comment_text,like_count,published_at
0,@shaiknasirofficial,1.9M\nDislikes 😂,0,2026-02-14T03:50:25Z
1,@Hirokojima-t1r,On 14 February 2025 the dislikes are 1.9 million,0,2026-02-13T19:37:07Z
2,@VincenzoS7S,1.9M dislikes,0,2026-02-11T22:45:34Z
3,@jetrho3447,kitne dislikes hai ??😂,0,2026-02-06T08:09:27Z
4,@Jospehsiju,You sicken me,0,2026-02-05T23:10:35Z
...,...,...,...,...
9995,@rockysingh837,Pleasw inko fully ignore karo indians,0,2020-08-15T02:38:02Z
9996,@AmanSingh-cb4qy,Job done,0,2020-08-15T02:37:57Z
9997,@ATR-u6e,Uparbala teri sath nehi hai pranati kahike,0,2020-08-15T02:37:46Z
9998,@kanusharmaphotography,Most bakwas.,0,2020-08-15T02:37:20Z


## Clean the data in the comment_text and remove emojis and non-ascii characters

In [21]:
import re

def clean_text(text):
    text = str(text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

In [22]:
df["comment_text"] = df["comment_text"].apply(clean_text)

In [23]:
df

,author,comment_text,like_count,published_at
0,@shaiknasirofficial,1.9M Dislikes,0,2026-02-14T03:50:25Z
1,@Hirokojima-t1r,On 14 February 2025 the dislikes are 1.9 million,0,2026-02-13T19:37:07Z
2,@VincenzoS7S,1.9M dislikes,0,2026-02-11T22:45:34Z
3,@jetrho3447,kitne dislikes hai ??,0,2026-02-06T08:09:27Z
4,@Jospehsiju,You sicken me,0,2026-02-05T23:10:35Z
...,...,...,...,...
9995,@rockysingh837,Pleasw inko fully ignore karo indians,0,2020-08-15T02:38:02Z
9996,@AmanSingh-cb4qy,Job done,0,2020-08-15T02:37:57Z
9997,@ATR-u6e,Uparbala teri sath nehi hai pranati kahike,0,2020-08-15T02:37:46Z
9998,@kanusharmaphotography,Most bakwas.,0,2020-08-15T02:37:20Z


## Change the date format to match the format to that of sql.

In [25]:
df["published_at"] = pd.to_datetime(df["published_at"])

In [27]:
df["published_at"] = df["published_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

In [28]:
df

,author,comment_text,like_count,published_at
0,@shaiknasirofficial,1.9M Dislikes,0,2026-02-14 03:50:25
1,@Hirokojima-t1r,On 14 February 2025 the dislikes are 1.9 million,0,2026-02-13 19:37:07
2,@VincenzoS7S,1.9M dislikes,0,2026-02-11 22:45:34
3,@jetrho3447,kitne dislikes hai ??,0,2026-02-06 08:09:27
4,@Jospehsiju,You sicken me,0,2026-02-05 23:10:35
...,...,...,...,...
9995,@rockysingh837,Pleasw inko fully ignore karo indians,0,2020-08-15 02:38:02
9996,@AmanSingh-cb4qy,Job done,0,2020-08-15 02:37:57
9997,@ATR-u6e,Uparbala teri sath nehi hai pranati kahike,0,2020-08-15 02:37:46
9998,@kanusharmaphotography,Most bakwas.,0,2020-08-15 02:37:20


# Save the data

In [37]:
df.to_csv("Youtube Abuse Analysis Comments.csv",index=False,encoding="utf-8")